# 📰 Fake News Detection — Complete Data Science Project
### Step-by-step: Data Loading → Cleaning → EDA → Model Training → Prediction
---
**Dataset:** 44,898 news articles (Fake + Real)  
**Goal:** Har news ki Fake/Real percentage detect karna with graphs


## 📦 Step 1: Libraries Install & Import

In [ ]:
# ── Install (agar pehli baar hai) ──
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re, math, warnings, os
from collections import Counter

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['text.color']       = '#e6edf3'
plt.rcParams['axes.labelcolor']  = '#8b949e'
plt.rcParams['xtick.color']      = '#8b949e'
plt.rcParams['ytick.color']      = '#8b949e'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['figure.dpi']       = 120

print("✅ Libraries successfully import ho gayi!")
print(f"   Pandas:  {pd.__version__}")
print(f"   Numpy:   {np.__version__}")


## 📂 Step 2: Data Load Karo

In [ ]:
# ── Paths — apna path yahan set karo ──
import os
BASE = os.path.dirname(os.path.abspath('__file__'))   # notebook ka folder
FAKE_PATH = os.path.join(BASE, 'Fake.csv')
TRUE_PATH = os.path.join(BASE, 'True.csv')

# ── Load ──
fake_df = pd.read_csv(FAKE_PATH)
true_df = pd.read_csv(TRUE_PATH)

# ── Label lagao ──
fake_df['label'] = 1   # 1 = FAKE
true_df['label'] = 0   # 0 = REAL

print("=" * 50)
print("📊 FAKE News Dataset:")
print(f"   Rows    : {fake_df.shape[0]:,}")
print(f"   Columns : {fake_df.shape[1]}")
print("\n📊 TRUE News Dataset:")
print(f"   Rows    : {true_df.shape[0]:,}")
print(f"   Columns : {true_df.shape[1]}")
print("=" * 50)

print("\n🔍 FAKE.csv columns:", list(fake_df.columns))
print("\nFAKE.csv — pehli 3 rows:")
fake_df.head(3)


## 🔗 Step 3: Dono Datasets Merge Karo

In [ ]:
# ── Combine karo aur shuffle karo ──
df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"✅ Combined Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nData types:")
print(df.dtypes)
print("\nPehli 5 rows:")
df.head()


## 📈 Step 4: Raw Dataset — Pehla Graph (Fake vs Real Distribution)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

GREEN, RED = '#2ecc71', '#e74c3c'
BLUE, ORANGE = '#3498db', '#f39c12'
TEXT, MUTED = '#e6edf3', '#8b949e'

counts = df['label'].value_counts().sort_index()
labels_pie = [f'Real\n{counts[0]:,}', f'Fake\n{counts[1]:,}']

# --- Pie chart ---
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    counts, labels=labels_pie, colors=[GREEN, RED],
    autopct='%1.1f%%', startangle=90, explode=(0.03, 0.03),
    wedgeprops=dict(edgecolor='#0d1117', linewidth=2.5),
    textprops=dict(color=TEXT, fontsize=11, fontweight='bold')
)
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold'); at.set_color('#0d1117')
ax1.set_title('Fake vs Real Distribution', color=TEXT, fontsize=13, fontweight='bold')

# --- Bar chart ---
ax2 = axes[1]
bars = ax2.bar(['Real News', 'Fake News'], counts.values,
               color=[GREEN, RED], edgecolor='#0d1117', width=0.5)
for bar, val in zip(bars, counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
             f'{val:,}', ha='center', va='bottom', color=TEXT,
             fontsize=12, fontweight='bold')
ax2.set_ylabel('Article Count', color=MUTED)
ax2.set_title('Article Count by Class', color=TEXT, fontsize=13, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.suptitle('📊 Dataset Overview — Before Cleaning', color=TEXT, fontsize=14,
             fontweight='bold', y=1.02)
plt.show()
print(f"\n📌 Total: {len(df):,} | Fake: {counts[1]:,} ({counts[1]/len(df)*100:.1f}%) | Real: {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")


## 🔍 Step 5: Null Values Check Karo

In [ ]:
print("=" * 50)
print("🔍 NULL VALUES (Missing Data)")
print("=" * 50)
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({'Null Count': null_counts, 'Percentage': null_pct})
print(null_df)
print(f"\nTotal null cells: {df.isnull().sum().sum()}")

# ── Graph: Null heatmap ──
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117')

null_matrix = df.isnull().astype(int)
sns.heatmap(null_matrix.T, ax=ax, cmap='RdYlGn_r',
            cbar=True, yticklabels=df.columns,
            xticklabels=False, linewidths=0)
ax.set_title('🔍 Null Values Heatmap (Green = OK, Red = NULL)', 
             color=TEXT, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT)
plt.tight_layout()
plt.show()


## 🧹 Step 6: Data Cleaning — Null, Duplicate, Empty Values Hatao

In [ ]:
print("BEFORE Cleaning:")
print(f"  Total rows       : {len(df):,}")
print(f"  Null values      : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")
print(f"  Empty title      : {(df['title'].str.strip() == '').sum()}")
print(f"  Empty text       : {(df['text'].str.strip() == '').sum()}")

# ── 1. Null fill / drop ──
df['title']   = df['title'].fillna('')
df['text']    = df['text'].fillna('')
df['subject'] = df['subject'].fillna('unknown')
df['date']    = df['date'].fillna('unknown')

# ── 2. Blank rows drop karo ──
df = df[df['title'].str.strip() != '']
df = df[df['text'].str.strip()  != '']

# ── 3. Duplicates hatao ──
before_dup = len(df)
df = df.drop_duplicates(subset=['title', 'text'])
removed_dup = before_dup - len(df)

# ── 4. Date parse ──
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df = df.reset_index(drop=True)

print("\nAFTER Cleaning:")
print(f"  Total rows       : {len(df):,}")
print(f"  Null values      : {df.isnull().sum().sum()}")
print(f"  Duplicates removed: {removed_dup}")
print(f"  Date parsed      : {df['date'].notna().sum():,} rows")
print("\n✅ Data clean ho gaya!")


## ⚙️ Step 7: Feature Engineering — Naye Columns Banao

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ── Features ──
df['content']    = df['title'] + ' ' + df['subject'] + ' ' + df['text']
df['clean_text'] = df['content'].apply(normalize_text)
df['word_count'] = df['clean_text'].str.split().str.len()
df['char_count'] = df['clean_text'].str.len()
df['title_len']  = df['title'].str.split().str.len()
df['has_reuters'] = df['text'].str.contains('reuters', case=False).astype(int)
df['has_trump']   = df['text'].str.contains('trump', case=False).astype(int)
df['has_obama']   = df['text'].str.contains('obama', case=False).astype(int)
df['year']       = df['date'].dt.year.fillna(0).astype(int)
df['month']      = df['date'].dt.month.fillna(0).astype(int)

print("✅ Naye features ban gaye:")
print(df[['title', 'word_count', 'char_count', 'title_len',
          'has_reuters', 'has_trump', 'year', 'label']].head(5))
print(f"\nTotal columns: {df.shape[1]}")


## 📊 Step 8: EDA — Word Count & Text Length Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')

fake = df[df['label'] == 1]
real = df[df['label'] == 0]

# ── 1. Word count histogram ──
ax = axes[0, 0]
ax.hist(real['word_count'].clip(0, 1500),  bins=60, color=GREEN,  alpha=0.75, label='Real', density=True)
ax.hist(fake['word_count'].clip(0, 1500),  bins=60, color=RED,    alpha=0.75, label='Fake', density=True)
ax.set_title('Word Count Distribution', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Word Count'); ax.set_ylabel('Density')
ax.legend(facecolor='#161b22', labelcolor=TEXT)

# ── 2. Char count boxplot ──
ax = axes[0, 1]
data_box = [real['char_count'].clip(0, 15000), fake['char_count'].clip(0, 15000)]
bp = ax.boxplot(data_box, labels=['Real', 'Fake'], patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor(GREEN); bp['boxes'][1].set_facecolor(RED)
for w in bp['whiskers'] + bp['caps']: w.set_color(MUTED)
for f in bp['fliers']: f.set(marker='.', color=MUTED, alpha=0.3)
ax.set_title('Char Count Boxplot', color=TEXT, fontsize=12, fontweight='bold')
ax.set_ylabel('Characters')

# ── 3. Title length ──
ax = axes[1, 0]
ax.hist(real['title_len'].clip(0, 30), bins=30, color=GREEN, alpha=0.75, label='Real', density=True)
ax.hist(fake['title_len'].clip(0, 30), bins=30, color=RED,   alpha=0.75, label='Fake', density=True)
ax.set_title('Title Word Count', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Title Word Count')
ax.legend(facecolor='#161b22', labelcolor=TEXT)

# ── 4. Avg word count bar ──
ax = axes[1, 1]
avg_wc = df.groupby('label')['word_count'].mean()
bars = ax.bar(['Real (0)', 'Fake (1)'], avg_wc.values, color=[GREEN, RED],
              edgecolor='#0d1117', width=0.5)
for b, v in zip(bars, avg_wc.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
            f'{v:.0f} words', ha='center', va='bottom', color=TEXT, fontsize=11, fontweight='bold')
ax.set_title('Average Word Count per Class', color=TEXT, fontsize=12, fontweight='bold')
ax.set_ylabel('Avg Words')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('📊 Text Length Analysis — Fake vs Real', color=TEXT, fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 📂 Step 9: Subject Category Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0d1117')

palette = ['#3498db','#2ecc71','#f39c12','#e74c3c','#9b59b6',
           '#1abc9c','#e67e22','#2980b9','#d35400','#8e44ad']

# ── All subjects ──
ax = axes[0]
subj_all = df['subject'].value_counts().head(10)
bars = ax.barh(subj_all.index[::-1], subj_all.values[::-1],
               color=palette[:len(subj_all)], edgecolor='#0d1117', height=0.7)
for b in bars:
    ax.text(b.get_width() + 100, b.get_y() + b.get_height()/2,
            f'{int(b.get_width()):,}', va='center', color=TEXT, fontsize=9)
ax.set_title('Top 10 Subjects (Overall)', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Count')

# ── Grouped by label ──
ax = axes[1]
subj_label = df.groupby(['subject', 'label']).size().unstack(fill_value=0)
subj_label = subj_label.loc[subj_all.index]
x = range(len(subj_label))
width = 0.38
bars_r = ax.bar([i - width/2 for i in x], subj_label[0], width=width,
                color=GREEN, alpha=0.9, label='Real', edgecolor='#0d1117')
bars_f = ax.bar([i + width/2 for i in x], subj_label[1], width=width,
                color=RED, alpha=0.9, label='Fake', edgecolor='#0d1117')
ax.set_xticks(list(x))
ax.set_xticklabels(subj_label.index, rotation=35, ha='right', fontsize=8.5)
ax.legend(facecolor='#161b22', labelcolor=TEXT)
ax.set_title('Fake vs Real by Subject', color=TEXT, fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('📂 Subject Distribution Analysis', color=TEXT, fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 📅 Step 10: Time-based Analysis (Year & Month)

In [ ]:
df_dated = df[df['year'] > 2010].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0d1117')

# ── By year ──
ax = axes[0]
year_label = df_dated.groupby(['year', 'label']).size().unstack(fill_value=0)
for col, color, lbl in zip([0, 1], [GREEN, RED], ['Real', 'Fake']):
    if col in year_label.columns:
        ax.plot(year_label.index, year_label[col], marker='o', color=color,
                linewidth=2.5, markersize=7, label=lbl)
ax.set_title('Articles per Year', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Count')
ax.legend(facecolor='#161b22', labelcolor=TEXT)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── By month ──
ax = axes[1]
df_2017 = df_dated[df_dated['year'].isin([2016, 2017, 2018])]
month_label = df_2017.groupby(['month', 'label']).size().unstack(fill_value=0)
mnths = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for col, color, lbl in zip([0, 1], [GREEN, RED], ['Real', 'Fake']):
    if col in month_label.columns:
        ax.bar([m - 0.2 + col*0.4 for m in month_label.index],
               month_label[col], width=0.38, color=color, alpha=0.85,
               label=lbl, edgecolor='#0d1117')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(mnths, fontsize=9)
ax.set_title('Articles by Month (2016-2018)', color=TEXT, fontsize=12, fontweight='bold')
ax.legend(facecolor='#161b22', labelcolor=TEXT)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('📅 Time-based Distribution', color=TEXT, fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🔑 Step 11: Keyword Features Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

features = [
    ('has_reuters', 'Reuters mention'),
    ('has_trump',   'Trump mention'),
    ('has_obama',   'Obama mention'),
]

for ax, (col, title) in zip(axes, features):
    grp = df.groupby(['label', col]).size().unstack(fill_value=0)
    x = [0, 1]
    width = 0.35
    for i, (lbl_val, color, name) in enumerate([(0, GREEN, 'Real'), (1, RED, 'Fake')]):
        vals = [grp.loc[lbl_val, 0] if 0 in grp.columns else 0,
                grp.loc[lbl_val, 1] if 1 in grp.columns else 0]
        ax.bar([xi + i*width - width/2 for xi in x], vals, width=width,
               color=color, alpha=0.9, label=name, edgecolor='#0d1117')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No', 'Yes'])
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold')
    ax.legend(facecolor='#161b22', labelcolor=TEXT, fontsize=8)
    ax.set_ylabel('Count')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('🔑 Keyword Presence in Fake vs Real News', color=TEXT,
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🔗 Step 12: Feature Correlation Heatmap

In [ ]:
num_cols = ['label', 'word_count', 'char_count', 'title_len',
            'has_reuters', 'has_trump', 'has_obama']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0d1117')
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, linecolor='#0d1117',
            annot_kws={'size': 10, 'color': 'black'})
ax.set_title('🔗 Feature Correlation Heatmap', color=TEXT, fontsize=13,
             fontweight='bold', pad=15)
ax.tick_params(colors=TEXT)
plt.tight_layout()
plt.show()


## 🤖 Step 13: Model Training (TF-IDF + Linear SVC)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score,
                             precision_score, recall_score)

# ── Split ──
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'], df['label'], test_size=0.30, stratify=df['label'], random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")

# ── Build models ──
def make_pipeline(estimator):
    return Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', max_features=50000,
                                  ngram_range=(1,2), min_df=2, sublinear_tf=True)),
        ('model', estimator)
    ])

models = {
    'Linear SVC':           make_pipeline(LinearSVC(C=1.0)),
    'Logistic Regression':  make_pipeline(LogisticRegression(max_iter=2000, n_jobs=-1)),
    'Multinomial NB':       make_pipeline(MultinomialNB(alpha=0.3)),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    results[name] = {
        'accuracy':  accuracy_score(y_val, preds),
        'f1':        f1_score(y_val, preds),
        'precision': precision_score(y_val, preds),
        'recall':    recall_score(y_val, preds),
    }
    print(f"✅ {name}: Accuracy={results[name]['accuracy']*100:.2f}%  F1={results[name]['f1']*100:.2f}%")

results_df = pd.DataFrame(results).T.sort_values('accuracy', ascending=False)
print("\n📊 Model Leaderboard:")
print(results_df.round(4))


## 📊 Step 14: Model Comparison Graph

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')

palette_m = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
metrics_to_plot = ['accuracy', 'f1', 'precision', 'recall']

# ── Grouped bar ──
ax = axes[0]
x = np.arange(len(results_df))
width = 0.2
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i*width, results_df[metric]*100, width=width,
           color=palette_m[i], alpha=0.9, label=metric.title(), edgecolor='#0d1117')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df.index, rotation=15, ha='right', fontsize=9)
ax.set_ylim(90, 101)
ax.set_ylabel('Score (%)')
ax.set_title('All Metrics Comparison', color=TEXT, fontsize=12, fontweight='bold')
ax.legend(facecolor='#161b22', labelcolor=TEXT, fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── Accuracy only horizontal ──
ax = axes[1]
colors_h = [palette_m[i] for i in range(len(results_df))]
bars = ax.barh(results_df.index[::-1], results_df['accuracy'][::-1]*100,
               color=colors_h, edgecolor='#0d1117', height=0.55)
for b, v in zip(bars, results_df['accuracy'][::-1]*100):
    ax.text(b.get_width() - 0.2, b.get_y() + b.get_height()/2,
            f'{v:.2f}%', va='center', ha='right', color='white',
            fontsize=10, fontweight='bold')
ax.set_xlim(90, 101)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Model Accuracy Ranking', color=TEXT, fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('🤖 Model Performance Comparison', color=TEXT, fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 🔲 Step 15: Confusion Matrix — Best Model

In [ ]:
best_name = results_df.index[0]
best_model = models[best_name]
print(f"✅ Best Model: {best_name}")

y_pred_test = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

# ── Heatmap ──
ax = axes[0]
im = ax.imshow(cm, cmap='RdYlGn', vmin=0)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred REAL','Pred FAKE'], color=TEXT, fontsize=11)
ax.set_yticklabels(['Act REAL','Act FAKE'],   color=TEXT, fontsize=11)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                color='black', fontsize=16, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Confusion Matrix\n({best_name})', color=TEXT, fontsize=12, fontweight='bold')

# ── Classification report bar ──
ax = axes[1]
report = classification_report(y_test, y_pred_test, target_names=['Real','Fake'],
                                output_dict=True)
classes = ['Real', 'Fake']
mets = ['precision', 'recall', 'f1-score']
colors_rep = [GREEN, RED]
x_pos = np.arange(len(mets))
width = 0.35
for i, (cls, color) in enumerate(zip(classes, colors_rep)):
    vals = [report[cls][m] * 100 for m in mets]
    bars = ax.bar(x_pos + i*width - width/2, vals, width=width,
                  color=color, alpha=0.9, label=cls, edgecolor='#0d1117')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
                f'{v:.1f}%', ha='center', va='bottom', color=TEXT, fontsize=8.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(['Precision','Recall','F1-Score'], fontsize=10)
ax.set_ylim(95, 103)
ax.legend(facecolor='#161b22', labelcolor=TEXT)
ax.set_title('Precision / Recall / F1', color=TEXT, fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('🔲 Best Model Evaluation', color=TEXT, fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Real','Fake']))


## 💬 Step 16: Top FAKE & REAL Terms (TF-IDF Weights)

In [ ]:
# Get top terms from best model (if it has coef_)
estimator = best_model.named_steps['model']
tfidf = best_model.named_steps['tfidf']

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor('#0d1117')

if hasattr(estimator, 'coef_'):
    feature_names = tfidf.get_feature_names_out()
    coefs = estimator.coef_[0]
    top_n = 15
    fake_idx = np.argsort(coefs)[-top_n:][::-1]
    real_idx = np.argsort(coefs)[:top_n]
    fake_terms = [(feature_names[i], coefs[i]) for i in fake_idx]
    real_terms = [(feature_names[i], abs(coefs[i])) for i in real_idx]

    # Fake terms
    ax = axes[0]
    terms_f, weights_f = zip(*fake_terms)
    bars = ax.barh(terms_f[::-1], weights_f[::-1], color=RED, alpha=0.85, edgecolor='#0d1117')
    ax.set_title('Top 15 FAKE News Terms', color=TEXT, fontsize=12, fontweight='bold')
    ax.set_xlabel('TF-IDF Weight')
    ax.tick_params(axis='y', labelsize=9)

    # Real terms
    ax = axes[1]
    terms_r, weights_r = zip(*real_terms)
    bars = ax.barh(terms_r[::-1], weights_r[::-1], color=GREEN, alpha=0.85, edgecolor='#0d1117')
    ax.set_title('Top 15 REAL News Terms', color=TEXT, fontsize=12, fontweight='bold')
    ax.set_xlabel('TF-IDF Weight (abs)')
    ax.tick_params(axis='y', labelsize=9)
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'Term weights not\navailable for this model',
                ha='center', va='center', color=TEXT, fontsize=12,
                transform=ax.transAxes)

plt.suptitle('💬 Most Influential Terms for Fake vs Real Detection',
             color=TEXT, fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 🔍 Step 17: Apni News Ko Analyze Karo — Percentage Graph

In [ ]:
def predict_news(title, text, subject='politics'):
    combined = normalize_text(f'{title} {subject} {text}')
    pred = int(best_model.predict([combined])[0])
    
    # Probability
    est = best_model.named_steps['model']
    if hasattr(est, 'predict_proba'):
        fake_prob = float(best_model.predict_proba([combined])[0][1])
    else:
        raw = float(best_model.decision_function([combined])[0])
        fake_prob = 1.0 / (1.0 + math.exp(-raw))
    
    real_prob = 1.0 - fake_prob
    label = 'FAKE' if pred == 1 else 'REAL'
    
    # ── Plot ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#0d1117')
    color = RED if label == 'FAKE' else GREEN
    
    # Donut
    ax = axes[0]
    sizes = [real_prob*100, fake_prob*100]
    colors_d = [GREEN, RED]
    wedges, texts, ats = ax.pie(
        sizes, labels=['REAL','FAKE'], colors=colors_d, explode=(0.03,0.03),
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='#0d1117', linewidth=3),
        textprops=dict(color='white', fontsize=11, fontweight='bold')
    )
    for at in ats: at.set_fontsize(13); at.set_fontweight('bold')
    ax.text(0,  0.08, label,                        ha='center', color=color, fontsize=18, fontweight='bold')
    ax.text(0, -0.18, f'{max(sizes):.1f}% confident', ha='center', color=MUTED, fontsize=10)
    ax.set_title('Prediction', color=TEXT, fontsize=12, fontweight='bold')

    # Confidence bar
    ax = axes[1]
    ax.barh([''], [real_prob*100], color=GREEN, height=0.5)
    ax.barh([''], [fake_prob*100], left=[real_prob*100], color=RED, height=0.5)
    ax.set_xlim(0,100)
    ax.text(real_prob*50,     0, f'REAL\n{real_prob*100:.1f}%', ha='center', va='center',
            color='white', fontsize=11, fontweight='bold')
    ax.text(real_prob*100 + fake_prob*50, 0, f'FAKE\n{fake_prob*100:.1f}%', ha='center', va='center',
            color='white', fontsize=11, fontweight='bold')
    ax.set_title('Confidence Bar', color=TEXT, fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Gauge
    ax = axes[2]
    theta = np.linspace(0, np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), color=MUTED, linewidth=2)
    # Color arcs
    t_real = np.linspace(0, np.pi*real_prob, 50)
    t_fake = np.linspace(np.pi*real_prob, np.pi, 50)
    ax.fill_between(np.cos(t_real), 0, np.sin(t_real), alpha=0.35, color=GREEN)
    ax.fill_between(np.cos(t_fake), 0, np.sin(t_fake), alpha=0.35, color=RED)
    needle_angle = np.pi * fake_prob
    ax.annotate('', xy=(0.6*np.cos(needle_angle), 0.6*np.sin(needle_angle)),
                xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.3, 1.2)
    ax.axis('off')
    ax.text(0, -0.2, f'Fake Risk: {fake_prob*100:.1f}%', ha='center', color=color,
            fontsize=12, fontweight='bold')
    ax.set_title('Fake Risk Gauge', color=TEXT, fontsize=12, fontweight='bold')
    
    plt.suptitle(f'📰  "{title[:55]}..."\n→ {label} NEWS',
                 color=color, fontsize=13, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*55}")
    print(f"  Prediction : {label}")
    print(f"  Real %     : {real_prob*100:.2f}%")
    print(f"  Fake %     : {fake_prob*100:.2f}%")
    print(f"{'='*55}")
    return label, real_prob*100, fake_prob*100

print("✅ Function ready! Neeche example chalao:")


## 🧪 Step 18: Examples — Aap Apni News Bhi Try Kar Sakte Ho!

In [ ]:
# ══ Example 1: FAKE news ══
predict_news(
    title   = "Trump Claims Election Was Stolen, Secret Plan Revealed!",
    text    = "Breaking news! Featured image via Getty. Obama scandal. "
              "GOP sources say America is under threat. Left-news. Video politics. "
              "President trump is amazing. Read more at this link.",
    subject = "politics"
)


In [ ]:
# ══ Example 2: REAL news ══
predict_news(
    title   = "Federal Reserve Raises Interest Rates by 25 Basis Points",
    text    = "Reuters — Washington. The Federal Reserve said Wednesday it raised "
              "interest rates at its November meeting. President Donald Trump's "
              "administration had pressured the central bank. The news conference "
              "was held on Thursday. Factbox: Full details below.",
    subject = "politicsNews"
)


In [ ]:
# ══ Apni koi bhi news yahan daalo ══
MY_TITLE   = "Apna headline yahan likhein"
MY_TEXT    = "Apna article yahan likhein..."
MY_SUBJECT = "politics"   # ya: worldnews, politicsNews, etc.

predict_news(MY_TITLE, MY_TEXT, MY_SUBJECT)


## 📊 Step 19: Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
from matplotlib.gridspec import GridSpec
gs = GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.40)

def sa(ax, title=''):
    ax.set_facecolor('#161b22')
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    if title: ax.set_title(title, color=TEXT, fontsize=10, fontweight='bold', pad=7)

# 1 Dataset pie
ax1 = fig.add_subplot(gs[0, 0]); sa(ax1, 'Dataset Split')
cnts = df['label'].value_counts().sort_index()
ax1.pie(cnts, labels=[f'Real\n{cnts[0]:,}', f'Fake\n{cnts[1]:,}'],
        colors=[GREEN, RED], autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='#0d1117'), textprops=dict(color=TEXT, fontsize=8))

# 2 Model accuracy
ax2 = fig.add_subplot(gs[0, 1]); sa(ax2, 'Model Accuracy')
sorted_r = results_df.sort_values('accuracy')
ax2.barh(sorted_r.index, sorted_r['accuracy']*100,
         color=['#3498db','#2ecc71','#e74c3c'][:len(sorted_r)], edgecolor='#0d1117', height=0.5)
ax2.set_xlim(90, 101)
for i, (idx, row) in enumerate(sorted_r.iterrows()):
    ax2.text(row['accuracy']*100 - 0.1, i, f"{row['accuracy']*100:.2f}%",
             va='center', ha='right', color='white', fontsize=8, fontweight='bold')

# 3 Word count dist
ax3 = fig.add_subplot(gs[0, 2]); sa(ax3, 'Word Count')
ax3.hist(df[df.label==0]['word_count'].clip(0,1500), bins=40, color=GREEN, alpha=0.7, label='Real', density=True)
ax3.hist(df[df.label==1]['word_count'].clip(0,1500), bins=40, color=RED,   alpha=0.7, label='Fake', density=True)
ax3.legend(facecolor='#161b22', labelcolor=TEXT, fontsize=7)
ax3.set_xlabel('Word Count')

# 4 Confusion matrix
ax4 = fig.add_subplot(gs[0, 3]); sa(ax4, 'Confusion Matrix')
im = ax4.imshow(cm, cmap='RdYlGn')
ax4.set_xticks([0,1]); ax4.set_yticks([0,1])
ax4.set_xticklabels(['P-Real','P-Fake'], color=TEXT, fontsize=8)
ax4.set_yticklabels(['A-Real','A-Fake'], color=TEXT, fontsize=8)
for i in range(2):
    for j in range(2):
        ax4.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                 color='black', fontsize=13, fontweight='bold')

# 5 Subjects
ax5 = fig.add_subplot(gs[1, 0:2]); sa(ax5, 'Subjects Distribution')
subj = df['subject'].value_counts().head(8)
palette = ['#3498db','#2ecc71','#f39c12','#e74c3c','#9b59b6','#1abc9c','#e67e22','#2980b9']
ax5.bar(subj.index, subj.values, color=palette[:len(subj)], edgecolor='#0d1117', width=0.7)
ax5.set_xticklabels(subj.index, rotation=30, ha='right', fontsize=8)
for i,(idx,val) in enumerate(subj.items()):
    ax5.text(i, val+50, f'{val:,}', ha='center', color=TEXT, fontsize=7.5)

# 6 Top fake terms
ax6 = fig.add_subplot(gs[1, 2]); sa(ax6, 'Top FAKE Terms')
if hasattr(estimator, 'coef_'):
    terms_plot = fake_terms[:10]
    ts, ws = zip(*terms_plot)
    ax6.barh(ts[::-1], ws[::-1], color=RED, alpha=0.85, edgecolor='#0d1117')
    ax6.tick_params(axis='y', labelsize=7.5)

# 7 Top real terms
ax7 = fig.add_subplot(gs[1, 3]); sa(ax7, 'Top REAL Terms')
if hasattr(estimator, 'coef_'):
    terms_plot = real_terms[:10]
    ts, ws = zip(*terms_plot)
    ax7.barh(ts[::-1], ws[::-1], color=GREEN, alpha=0.85, edgecolor='#0d1117')
    ax7.tick_params(axis='y', labelsize=7.5)

# Stats text
fig.text(0.5, 0.98,
         f"📰 Fake News Detection — Final Dashboard  |  "
         f"Total: {len(df):,}  |  Fake: {cnts[1]:,}  |  Real: {cnts[0]:,}  |  "
         f"Best Model: {best_name}  |  Test Acc: {accuracy_score(y_test, y_pred_test)*100:.2f}%",
         ha='center', color=TEXT, fontsize=11, fontweight='bold')
plt.show()
print("\n✅ Complete Data Science Project Done!")
